In [1]:
from _setup import setup_project_root
PROJECT_ROOT = setup_project_root()
PROJECT_ROOT

WindowsPath('C:/Users/huyy/AirPollutionPrediction-CNN-BiLSTM')

In [2]:
import pandas as pd
from models.lgbm import LGBMBaseConfig, build_xy, fit_model, predict
from evaluation.metrics import compute_metrics, compute_metrics_real_scale
import joblib

In [3]:
train_path = PROJECT_ROOT / "data" / "station_split" / "train.csv"
val_path   = PROJECT_ROOT / "data" / "station_split" / "val.csv"
test_path  = PROJECT_ROOT / "data" / "station_split" / "test.csv"

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

In [4]:
print(len(train_df), len(val_df), len(test_df))
print("Train stations:", train_df["Station_No"].nunique())
print("Test stations:",  test_df["Station_No"].nunique())
print("Overlap stations train-test:", set(train_df["Station_No"]).intersection(set(test_df["Station_No"])))

55611 6947 6957
Train stations: 6
Test stations: 6
Overlap stations train-test: {1, 2, 3, 4, 5, 6}


In [5]:
pm25_scaler = joblib.load(PROJECT_ROOT / "artifacts" / "pm25_scaler.pkl")

In [6]:
cfg = LGBMBaseConfig(
    time_col="date",
    target_col="PM2.5",
    lags=0,
    horizon=24,
    add_hour_feature=False,
    lgbm_params=dict(
        objective="regression",
        n_estimators=800,
        learning_rate=0.05,
        max_depth=8,
        num_leaves=63,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
    )
)

In [7]:
extra_feature_cols = [
    "TSP","O3","CO","NO2","SO2","Temperature","Humidity",
    "hour","day_of_week","hour_sin","hour_cos",
    "PM2.5_lag1","PM2.5_roll3"
]

In [8]:
X_train, y_train = build_xy(train_df, cfg, extra_feature_cols=extra_feature_cols)
X_val,   y_val   = build_xy(val_df,   cfg, extra_feature_cols=extra_feature_cols)
X_test,  y_test  = build_xy(test_df,  cfg, extra_feature_cols=extra_feature_cols)

In [10]:
model = fit_model(X_train, y_train, cfg=cfg)
pred_test = predict(model, X_test)

In [11]:
metrics_station_split = compute_metrics_real_scale(
    y_test, pred_test, scaler=pm25_scaler, mape_threshold=1.0
)
metrics_station_split

{'Correlation': 0.4283800892041336,
 'RMSE': 63.655549753771936,
 'MAPE': 0.4754175875821795,
 'MAE': 47.12117653137772}

In [12]:
metrics_station_split_1step = compute_metrics_real_scale(
    y_test[:,0], pred_test[:,0], scaler=pm25_scaler, mape_threshold=1.0
)
metrics_station_split_1step


{'Correlation': 0.5122420297845273,
 'RMSE': 59.80308648207016,
 'MAPE': 0.42890948065425677,
 'MAE': 43.37866532173552}

Khi train theo station:

mô hình học pattern PM2.5 đồng nhất hơn

giảm “nhiễu không gian” giữa các station

Global model:

phải học trung bình hóa hành vi 6 station

làm mờ các quan hệ cục bộ

In [14]:
import numpy as np
import pandas as pd

# Align station_ids theo X_test.index (tránh mismatch)
station_ids = test_df.loc[X_test.index, "Station_No"].to_numpy()

rows = []
for sid in np.unique(station_ids):
    m = station_ids == sid
    if m.sum() < 50:
        continue
    met = compute_metrics_real_scale(
        y_test[m,0], pred_test[m,0], scaler=pm25_scaler, mape_threshold=1.0
    )
    rows.append({"Station_No": sid, "n": int(m.sum()), **met})

df_station_metrics = pd.DataFrame(rows).sort_values("RMSE")
df_station_metrics.head(10), df_station_metrics.tail(10)


(   Station_No     n  Correlation       RMSE      MAPE        MAE
 3           4  1151     0.534847  56.267366  0.448254  41.554191
 4           5  1151     0.551274  57.456805  0.394483  41.829633
 0           1  1152     0.507811  58.087424  0.376337  41.947969
 1           2  1152     0.483097  61.875894  0.470650  44.947787
 2           3  1152     0.493583  62.181194  0.426502  44.107696
 5           6  1151     0.501110  62.622539  0.457507  45.883963,
    Station_No     n  Correlation       RMSE      MAPE        MAE
 3           4  1151     0.534847  56.267366  0.448254  41.554191
 4           5  1151     0.551274  57.456805  0.394483  41.829633
 0           1  1152     0.507811  58.087424  0.376337  41.947969
 1           2  1152     0.483097  61.875894  0.470650  44.947787
 2           3  1152     0.493583  62.181194  0.426502  44.107696
 5           6  1151     0.501110  62.622539  0.457507  45.883963)

Phân tích chi tiết theo nhóm station
 Nhóm station học TỐT (4 & 5)

Correlation > 0.53

RMSE ≈ 56–57, MAE ≈ 41–42

Đặc điểm suy luận hợp lý:

biến động PM2.5 ổn định hơn

ít spike cực đoan

pattern lặp lại rõ

LGBM phát huy tốt khi chuỗi có tính “đều”.

Nhóm station mức TRUNG BÌNH (1, 3, 6)

Correlation ~0.49–0.51

RMSE ~58–63

MAE ~42–46

Có:

spike

biến động ngắn hạn

Model học được xu hướng, nhưng sai số tăng khi biến động mạnh.

Station kém nhất (2)

Correlation thấp nhất (0.48)

MAE cao

MAPE cao nhất (~47%)

Cho thấy:

PM2.5 tại station này rất khó dự đoán

có thể chịu ảnh hưởng mạnh bởi:

giao thông cục bộ

nguồn thải đột ngột

yếu tố ngoại sinh không có trong feature